# Leakage Audit / 数据泄漏审计

这个 notebook 回答一个问题：**清洗之后，数据里还剩下多少"不用理解内容就能分对"的捷径？**

`02_lstm_corrected` 删掉了 Reuters 署名和 `subject` 列，但只删了真新闻一侧的痕迹。本 notebook 使用新的统一数据管线 `src/data.py`（对称清洗 + 去重 + 按组划分），并用三种方法检查清洗效果：

1. **残留痕迹统计**：各种来源格式痕迹在真 / 假新闻中出现的比例，清洗前后对比
2. **划分完整性检查**：val / test 里有没有和 train 重复或近似重复的文章
3. **捷径探针**：用最简单的 TF-IDF + 逻辑回归，只给它看标题、只看开头 20 个词，或者只看标点和大小写"形状"，看它能分得多准

> ⚠️ 本 notebook **只使用 train 和 val**，完全不碰 test。test 必须留到最终评估时只用一次。

This notebook asks one question: **after cleaning, how many shortcuts — ways to classify correctly without understanding content — are left in the data?**

`02_lstm_corrected` removed the Reuters byline and the `subject` column, but only cleaned the real-news side. Here we use the new shared pipeline `src/data.py` (symmetric cleaning + de-duplication + grouped split) and audit it three ways: residual-artefact rates, split integrity, and shortcut probes. **Only train and val are used — the test set is never touched.**

## 1. Setup / 环境

In [1]:
import sys
from pathlib import Path

import pandas as pd

# 让 notebook 能 import 项目根目录下的 src/ / make src/ importable from notebooks/
sys.path.insert(0, str(Path.cwd().parent))

from src.utils import set_seed
from src.data import load_dataset, load_report, load_raw, get_splits
from src import probes

set_seed(42)
pd.set_option('display.max_colwidth', 120)

/Users/sl1425/Library/Python/3.9/lib/python/site-packages/urllib3/__init__.py:35: NotOpenSSLWarning: urllib3 v2 only supports OpenSSL 1.1.1+, currently the 'ssl' module is compiled with 'LibreSSL 2.8.3'. See: https://github.com/urllib3/urllib3/issues/3020
  warnings.warn(


## 2. Load data & cleaning report / 加载数据与清洗统计

`load_dataset()` 第一次运行时会从原始 CSV 构建数据集（约 2 分钟），之后直接读缓存。清洗规则或原始数据一旦改变，缓存会自动失效并重新构建。

`load_dataset()` builds from the raw CSVs on first run (~2 min) and reads a cache afterwards; editing the rules or the raw data invalidates the cache automatically.

In [2]:
df = load_dataset()
raw = load_raw()
report = load_report()

steps = pd.DataFrame({
    'fake': [report['raw_by_label']['fake'],
             -report['removed_short_body_by_label']['fake'],
             -report['removed_exact_duplicates_by_label']['fake']],
    'real': [report['raw_by_label']['real'],
             -report['removed_short_body_by_label']['real'],
             -report['removed_exact_duplicates_by_label']['real']],
}, index=['原始 / raw',
          '正文 < 20 词 / body < 20 words',
          '完全重复 / exact duplicates'])
steps.loc['最终 / final'] = steps.sum()
steps

Loaded cached dataset: 37,982 rows (clean_dataset.csv)


,fake,real
原始 / raw,23481,21417
正文 < 20 词 / body < 20 words,-1179,-7
完全重复 / exact duplicates,-5242,-488
最终 / final,17060,20922


In [3]:
print(f"Conflicting-label duplicates removed: {report['removed_conflicting_label_rows']}")
print(f"Near-duplicate groups (2+ docs):       {report['near_dup_groups_multi_doc']:,} "
      f"({report['docs_in_multi_doc_groups']:,} docs, largest = {report['largest_group_size']})")
print(f"Groups mixing fake and real:           {report['mixed_label_groups']}")
print()
print(df.groupby('split')['label'].agg(n='size', real_share='mean').round(3))

Conflicting-label duplicates removed: 0
Near-duplicate groups (2+ docs):       491 (1,218 docs, largest = 42)
Groups mixing fake and real:           6

           n  real_share
split                   
test    7630       0.550
train  24278       0.551
val     6074       0.550


几点说明 / Notes:

- **短正文过滤只看正文、不看标题。** 被删的几乎都是假新闻（只有标题、正文为空的视频帖）。如果保留它们，"正文很短"本身就会成为一个假新闻信号。
- **完全重复主要来自假新闻**：同一篇文章被多个假新闻网站转载，或者在不同栏目下重复发布。
- **近似重复不删除，而是分组**：同一组的文章只会进入同一个集合（train / val / test 之一）。
- 三个集合中真新闻的比例都约为 55%，说明分层划分生效了。

- **The short-body filter looks at the body only.** Almost all removed rows are fake headline-only video posts; keeping them would make "short body" a fake-news signal by itself.
- **Exact duplicates are mostly fake** (the same story re-posted across sites/sections).
- **Near-duplicates are grouped, not deleted**: a group lives in exactly one split.
- The real-news share is ~55% in every split, so stratification worked.

## 3. Before vs after: examples / 清洗前后示例

各取一篇真新闻和一篇假新闻，对比原始文本和清洗后的文本。注意看开头的署名和结尾的图片说明。

One real and one fake article, raw vs cleaned — watch the dateline at the start and the image credit at the end.

In [4]:
raw_by_id = raw.set_index('doc_id')
train = df[df['split'] == 'train']

def show(doc_id, n=400):
    r, c = raw_by_id.loc[doc_id], df.set_index('doc_id').loc[doc_id]
    print(f'=== {doc_id} (label={c.label}) ===')
    print('RAW   start:', repr(r.text[:n // 2]))
    print('CLEAN start:', repr(c.text[:n // 2]))
    print('RAW   end:  ', repr(r.text[-n // 2:]))
    print('CLEAN end:  ', repr(c.text[-n // 2:]))
    print()

# 确定性地挑选示例：第一篇带 "(Reporting by" 的真新闻、第一篇带 "Featured image via" 的假新闻
# Deterministic picks: first real article with "(Reporting by", first fake with "Featured image via"
real_id = next(i for i in train.doc_id if i.startswith('true') and '(Reporting by' in raw_by_id.loc[i, 'text'])
fake_id = next(i for i in train.doc_id if i.startswith('fake') and 'Featured image via' in raw_by_id.loc[i, 'text'])
show(real_id)
show(fake_id)

=== true_00884 (label=1) ===
RAW   start: ' WASHINGTON (Reuters) - When President Donald Trump announced on Oct. 12 he would stop subsidies to health insurers critical to the Obamacare law, the response from Democratic state officials was swif'
CLEAN start: 'When President Donald Trump announced on Oct. 12 he would stop subsidies to health insurers critical to the Obamacare law, the response from Democratic state officials was swift and predictable: a law'
RAW   end:   'energy, environment, healthcare, immigration and the economy at The Trump Effect [www.reuters.com/trump-effect]  \xa0(Reporting by Lawrence Hurley and Andrew Chung; Editing by Sue Horton and Will Dunham)'
CLEAN end:   'e Vince Chhabria asked rhetorically during a hearing: why "have all these attorneys general rushed to court seeking an emergency ruling against President Trump?" Chhabria then ruled in favor of Trump.'

=== fake_00007 (label=0) ===
RAW   start: 'In the wake of yet another court decision that derailed Dona

## 4. Residual artefacts / 残留痕迹统计

每一行是一种 **只和来源网站有关、和内容真假无关** 的格式特征，数值是含有该特征的文章比例（%）。左边是原始数据，右边是清洗后。理想情况下，清洗后的真假两列应该接近相等。

`make_raw_view` 用的是 *同一批文章、同一个划分*，只是把文本换回原始版本，所以前后差异只来自清洗本身。

Each row is a formatting feature tied to the **source site, not truthfulness**; values are % of articles containing it. `make_raw_view` keeps the *same articles and split* with raw text, so differences come from cleaning alone.

In [5]:
raw_view = probes.make_raw_view(raw, df)

artefacts = pd.concat({'RAW': probes.artefact_rates(raw_view),
                       'CLEANED': probes.artefact_rates(df)}, axis=1)
artefacts

RAW        CLEANED       
                              fake % real %  fake % real %
Reuters 字样 / "Reuters"           1.2   99.8     0.0    0.1
开头署名 / dateline "(Reuters) -"    0.0   98.5     0.0    0.0
空括号 / empty "()"                 0.3    0.0     0.0    0.0
记者署名 / "(Reporting by"           0.0    1.2     0.0    0.0
图片来源 / "Featured image"         51.4    0.0     0.0    0.0
Getty Images                    23.8    0.0     0.1    0.0
转载来源 / "Via:" / "Read more"     20.4    0.0     0.0    0.0
推文链接 / pic.twitter.com          15.5    0.0     0.0    0.0
URL                             14.5    0.2     0.0    0.0
@用户名 / @handle                  27.3    1.3     0.0    0.0
弯引号 / curly quotes               0.0   51.4     0.0    0.0
不换行空格 / \xa0                     0.0    5.2     0.0    0.0
句号后缺空格 / "word.Word"            90.3    0.5     0.0    0.0
标题方括号 / title "[...]"           16.2    0.0     0.0    0.0
标题 VIDEO / WATCH                38.7    0.3     3.0    0.3
标题感叹号 / title "!"               10.4    0.1     8.4    0.1
文风 / "said"                     57.7   93.7    57.7   93.7
文风 / "U.S."                     16.6   60.6    16.6   60.2

In [6]:
print('Titles with >=30% ALL-CAPS words')
print('  raw:    ', probes.all_caps_title_rate(raw_view))
print('  cleaned:', probes.all_caps_title_rate(df))

Titles with >=30% ALL-CAPS words
  raw:     {'fake %': 18.7, 'real %': 0.0}
  cleaned: {'fake %': 16.8, 'real %': 0.1}


怎么读这张表 / How to read the table:

- **格式痕迹**（署名、图片说明、链接、弯引号、句号后缺空格等）清洗后应接近 0。
- **标题感叹号和全大写**在清洗后的 `title` 列中仍然存在，这是故意的：`title` / `content` 保留原样供人工阅读，而模型读取的 `model_text` 会统一转小写、去掉标点，所以这两类信号不会进入模型。
- **"said" 和 "U.S."** 是文风差异，不是格式痕迹（路透社几乎每句都用 "said"），规则清洗不能也不应该删除。它们会出现在后面的探针结果里。

- **Formatting artefacts** should be ~0 after cleaning.
- **Title "!" and ALL-CAPS** remain in the human-readable `title` column on purpose; the models read `model_text`, which is lower-cased with punctuation removed, so these never reach them.
- **"said" / "U.S."** are writing *style*, not formatting — rules cannot (and should not) remove them. They reappear in the probe results below.

## 5. Split integrity / 划分完整性检查

两项检查 / Two checks:

1. 有没有近似重复组同时出现在多个集合里？**必须为 0。**
2. val / test 中有多少文章和 train 共享某个"指纹"：
   - 完全相同 / 标题相同 / 开头相同：管线直接处理，**必须为 0**
   - 结尾相同：管线也用它分组，但会跳过被超过 25 篇共用的结尾（固定页脚），所以可能有极少数
   - 中间相同：管线 *没有* 用它分组，是**独立的交叉检验**

In [7]:
print('Near-duplicate groups spanning more than one split:', probes.groups_cross_splits(df))
probes.split_overlap(df)

Near-duplicate groups spanning more than one split: 0


,val,test
完全相同 / exact model_text,0,0
标题相同 / same title,0,0
开头 25 词相同 / same lead,0,0
结尾 25 词相同 / same tail,0,0
中间 25 词相同 / same middle (independent check),2,4


## 6. Shortcut probes / 捷径探针

每个探针都是 **在 train 上训练、在 val 上评估** 的 TF-IDF + 逻辑回归。各行的区别只在于给模型看什么：

| 探针 | 模型能看到什么 | 如果分数很高，说明…… |
|---|---|---|
| 完整模型输入 | `model_text`（深度模型实际读取的文本） | 简单的词袋模型就能做到，深度模型的高分不能说明它更"懂" |
| 只看标题 | 标题 | 标题的写法就暴露了来源 |
| 正文前 / 后 20 词 | 开头或结尾的 20 个词 | 开头 / 结尾残留了来源格式 |
| 形状（保留大小写标点） | 字母全换成 x / X，只剩标点和大小写 | 排版格式暴露了来源（会影响区分大小写、读取原始文本的模型） |
| 形状（模型输入） | `model_text` 的形状：只剩词长和数字 | 连词长分布都能区分来源 |

左边是原始文本，右边是清洗后。**大约需要 5 分钟。**

Each probe is TF-IDF + logistic regression **fit on train, scored on val**; rows differ only in what the model is allowed to see. Left = raw text, right = cleaned. **Takes ~5 minutes.**

In [8]:
train, val, _ = get_splits(df)          # 第三个是 test，这里故意不用 / the third is test — deliberately unused
raw_train = raw_view[raw_view['split'] == 'train']
raw_val = raw_view[raw_view['split'] == 'val']

probe_results = pd.concat({'RAW': probes.run_probe_suite(raw_train, raw_val),
                           'CLEANED': probes.run_probe_suite(train, val)}, axis=1)
probe_results

RAW           CLEANED       
                              accuracy %    AUC accuracy %    AUC
完整模型输入 / full model_text            99.2  0.999       98.3  0.998
只看标题 / title only                   95.0  0.989       94.1  0.984
只看正文前 20 词 / body first 20          99.7  1.000       94.6  0.987
只看正文后 20 词 / body last 20           94.5  0.986       86.8  0.936
形状（保留大小写标点）/ shape of content      100.0  1.000       95.5  0.992
形状（模型输入）/ shape of model_text       77.8  0.840       74.1  0.798

## 7. What does the probe rely on? / 探针依赖哪些特征？

在清洗后的完整输入上，逻辑回归权重最大的 25 个特征。清洗前，这里排在前面的是 `reuters`、`featured image`、`via`、`getty` 这类来源痕迹；如果清洗有效，剩下的应该是文风和话题用词。

The 25 largest-weight features on the cleaned full input. Before cleaning, source artefacts such as `reuters`, `featured image`, `via`, `getty` dominated; if cleaning worked, only style and topic words should remain.

In [9]:
_, vectorizer, clf = probes.run_probe(train, val, lambda d: d['model_text'])
probes.top_features(vectorizer, clf, k=25)

,→ real (label 1),→ fake (label 0)
0,said,this
1,said on,president trump
2,us,just
3,on wednesday,gop
4,on tuesday,obama
5,us president,hillary
6,on,mr
7,on thursday,watch
8,president donald,here
9,said in,sen


## 8. Conclusions / 结论

### 清洗解决了什么 / What cleaning fixed

- **格式痕迹基本清零。** 原始数据中，98.5% 的真新闻以 "(Reuters) -" 开头，51.4% 的假新闻带 "Featured image" 图片说明，90.3% 的假新闻有 "word.Word" 这种句号后缺空格的爬取痕迹，51.4% 的真新闻含弯引号。清洗后这些都降到 0–0.1%。
- **重复泄漏为 0。** 删除了 5,730 篇完全重复的文章，又把 1,218 篇近似重复的文章分成 491 组。没有任何一组跨越 train / val / test；val 和 test 中没有一篇与 train 完全相同，也没有标题、开头或结尾相同的文章。独立的"中间段"检查只发现 2 + 4 篇（占 val + test 的 0.04%）：5 篇是路透社在同一事件的不同稿件中复用的背景段落（其中 "Russia casts 10th U.N. veto" 那一对可以算作同一事件的两个版本），1 篇是同一档节目的系列帖子。影响可以忽略。
- **探针依赖的特征变了。** 清洗前排在前面的是 `reuters`、`featured image`、`via`、`getty` 这类来源痕迹；清洗后只剩文风和话题用词。

- **Formatting artefacts are essentially gone**: the Reuters dateline (98.5% of real), "Featured image" (51.4% of fake), missing-space scrape artefacts (90.3% of fake) and curly quotes (51.4% of real) all drop to 0–0.1%.
- **Duplicate leakage is zero**: 5,730 exact duplicates removed; 1,218 near-duplicates grouped into 491 groups; no group spans two splits, and no val/test article shares its text, title, lead or tail with train. The independent middle-window check finds only 2 + 4 articles (0.04% of val + test): five reuse a Reuters background paragraph across stories on the same event (one pair is arguably two versions of the same story), one is a same-show series post — negligible.
- **The probe's top features changed** from source artefacts to style and topic words.

### 清洗解决不了什么 / What cleaning cannot fix

清洗后的探针结果（val 集）/ Probe results after cleaning (val):

| 模型只能看到 / The model sees only | Accuracy |
|---|---|
| 完整模型输入 / full `model_text` | **98.3%** |
| 正文前 20 个词 / first 20 body words | 94.6% |
| 标题 / title | 94.1% |
| 正文后 20 个词 / last 20 body words | 86.8% |

**一个简单的词袋模型，只看标题或开头 20 个词，仍然有约 94% 的准确率。** 现在它依赖的是 **文风**，而不是格式：

- 真新闻侧是路透社的写作规范：`said`、`said on`、`on wednesday`、`nov`（"Nov. 8" 这种日期写法）、`spokesman`、`told`
- 假新闻侧是博客 / 评论的口吻：`this`、`just`、`you`、`here`、`why`、`even`，以及 `mr`、`sen`、`rep`、`gop` 这类缩写

这些是写作风格和话题，不是可以用正则删除的格式。如果强行删掉 "said" 或星期几，就等于在删改新闻内容本身。

**A bag-of-words model that sees only the title, or only the first 20 words, still reaches ~94%.** It now relies on **writing style**: Reuters house style on the real side (`said`, `on wednesday`, `nov`, `spokesman`) versus a blog / opinion voice on the fake side (`this`, `just`, `you`, `why`, `mr`, `sen`, `gop`). That is style and topic, not formatting — deleting it with rules would mean editing the news content itself.

### 这对后续实验意味着什么 / What this means for the next experiments

1. **在 ISOT 上，任何模型的高准确率主要衡量的是"能不能分辨路透社文风和博客文风"，不是"能不能分辨真假"。** 这个限制来自数据集的构造方式（真新闻全部来自一个来源，假新闻全部来自另一批来源），任何清洗都无法消除。要回答"模型是否学会了判断真假"，必须做跨数据集测试（阶段 5）。
2. **TF-IDF + 逻辑回归（val 98.3%）是 BiLSTM 和 DistilBERT 必须超过的基线。** 如果深度模型达不到这个水平，它的复杂度就没有带来任何价值。
3. **所有模型都应读取 `model_text`，而不是 `content`。** 保留大小写和标点的"形状"探针仍有 95.5%；`model_text` 统一转小写并去掉标点后，形状探针降到 74.1%。DistilBERT 也要用 `model_text`，否则它会比 LSTM 多看到标点和大小写这类来源信号，对比就不公平。
4. **这里的数字不能和 `02` / `03` notebook 的旧结果直接比较。** 数据、清洗方式和划分都不同。BiLSTM 和 DistilBERT 需要用新的数据管线重新训练（阶段 3、4）。

1. **On ISOT, high accuracy mostly measures "Reuters style vs blog style", not truthfulness.** This comes from how the dataset was built and no cleaning can remove it; answering the truthfulness question requires cross-dataset evaluation (phase 5).
2. **TF-IDF + logistic regression (val 98.3%) is the baseline** BiLSTM and DistilBERT must beat; otherwise their complexity adds nothing.
3. **Every model should read `model_text`, not `content`.** The shape probe drops from 95.5% (case + punctuation kept) to 74.1% on `model_text`. DistilBERT must use it too, or it sees source signals the LSTM cannot.
4. **These numbers are not comparable with the old `02` / `03` results** (different data, cleaning and split); both models must be retrained on this pipeline (phases 3–4).

### 在其他 notebook 中使用 / Using the pipeline in other notebooks

```python
import sys; from pathlib import Path
sys.path.insert(0, str(Path.cwd().parent))
from src.utils import set_seed
from src.data import load_dataset, get_splits

set_seed(42)
train, val, test = get_splits(load_dataset())
X_train, y_train = train['model_text'], train['label']   # 模型输入 / model input
# val：调参和 EarlyStopping / tuning and early stopping
# test：只在最终评估时用一次 / final evaluation only — once
```